# 09 存活分析 — 參考解答

松柏護理之家退伍軍人症群聚事件存活分析練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["death_date"] = pd.to_datetime(df["death_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

cases = df[df["infected"] == 1].copy()
cases["event"] = (cases["outcome"] == "dead").astype(int)

investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days

## 題目 1：CHF（心衰竭）的存活分析

In [ ]:
# KM 曲線：CHF vs No CHF
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("CHF", cases["comorbidity_chf"] == 1),
                     ("No CHF", cases["comorbidity_chf"] == 0)]:
    sub = cases[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("存活曲線：CHF vs No CHF")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank 檢定
chf_yes = cases[cases["comorbidity_chf"] == 1]
chf_no = cases[cases["comorbidity_chf"] == 0]

result = logrank_test(
    chf_yes["time_to_event"], chf_no["time_to_event"],
    event_observed_A=chf_yes["event"],
    event_observed_B=chf_no["event"],
)

print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

if result.p_value < 0.05:
    print("\n\u2192 p < 0.05：CHF 顯著影響存活")
else:
    print("\n\u2192 p \u2265 0.05：CHF 對存活的影響未達統計顯著")
    print("\u2192 可能因為樣本數不足（死亡人數僅 19），統計檢定力不夠")

## 題目 2：年齡分組的存活比較

In [ ]:
# 年齡分組
cases["age_group"] = np.where(cases["age"] >= 75, "\u226575", "<75")

fig, ax = plt.subplots(figsize=(8, 5))

for label in ["\u226575", "<75"]:
    sub = cases[cases["age_group"] == label]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"Age {label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("存活曲線：Age \u226575 vs <75")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank
old = cases[cases["age"] >= 75]
young = cases[cases["age"] < 75]

result_age = logrank_test(
    old["time_to_event"], young["time_to_event"],
    event_observed_A=old["event"],
    event_observed_B=young["event"],
)

print(f"Log-rank test statistic = {result_age.test_statistic:.3f}")
print(f"p-value = {result_age.p_value:.4f}")

if result_age.p_value < 0.05:
    print("\n\u2192 高齡（\u226575）組的存活顯著較差")
else:
    print("\n\u2192 年齡分組對存活的影響未達統計顯著")
    print("\u2192 護理之家住民普遍年齡較高，組間差異可能不夠大")

## 題目 3（挑戰題）：住院 vs 未住院 + Cox 迴歸

In [ ]:
# KM 曲線：住院 vs 未住院
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("住院", cases["hospitalized"] == 1),
                     ("未住院", cases["hospitalized"] == 0)]:
    sub = cases[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("存活曲線：住院 vs 未住院")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank
hosp_yes = cases[cases["hospitalized"] == 1]
hosp_no = cases[cases["hospitalized"] == 0]

result_hosp = logrank_test(
    hosp_yes["time_to_event"], hosp_no["time_to_event"],
    event_observed_A=hosp_yes["event"],
    event_observed_B=hosp_no["event"],
)

print(f"Log-rank test statistic = {result_hosp.test_statistic:.3f}")
print(f"p-value = {result_hosp.p_value:.4f}")

In [ ]:
# Cox 迴歸
cox_df = cases[[
    "time_to_event", "event", "age", "sex",
    "hospitalized", "comorbidity_copd", "comorbidity_chf",
]].copy()
cox_df["is_male"] = (cox_df["sex"] == "M").astype(int)
cox_df = cox_df.drop(columns=["sex"])

cph = CoxPHFitter()
cph.fit(cox_df, duration_col="time_to_event", event_col="event")

print("=== Cox 迴歸結果 ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

# HR 森林圖
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Cox Regression \u2014 HR 森林圖")
plt.tight_layout()
plt.show()

print("\n=== 解讀 ===")
hosp_hr = cph.summary.loc["hospitalized", "exp(coef)"]
print(f"hospitalized HR = {hosp_hr:.3f}")
if hosp_hr > 1:
    print("\u2192 住院者 HR > 1，看似住院『增加』死亡風險")
    print("\u2192 但這不代表住院是危險因子！")
    print("\u2192 住院是因為病情嚴重，是 confounding by indication")
    print("\u2192 住院是嚴重度的『標記』，不是死亡的『原因』")
else:
    print("\u2192 住院者 HR < 1，調整其他因子後住院可能有保護效果")
    print("\u2192 但解讀仍需小心 confounding by indication")

### 解讀重點

- **CHF**：心衰竭可能增加死亡風險，但在小樣本中可能未達統計顯著
- **年齡**：護理之家住民年齡普遍偏高，組間差異可能不大
- **住院**：這是存活分析中經典的 **confounding by indication** 案例
  - 住院者的死亡率可能較高，但原因是「比較嚴重的人才會住院」
  - 住院本身是治療行為，應該降低死亡風險
  - 但在觀察性資料中，住院的 HR 可能 > 1，因為它是嚴重度的標記
- **限制**：本案僅 19 例死亡，Cox 模型中放太多變項容易過度配適（overfitting），建議每個事件至少 10 個，所以最多放 1-2 個變項較穩定

## 題目 4 解答

In [ ]:
# 資料：結核病 (TB) 治療世代 -- 比較抗藥性 (MDR-TB) vs 非抗藥性病人達到痊癒的時間
rng = np.random.default_rng(409)
n = 400

drug_resistant = rng.binomial(1, 0.2, size=n)  # 20% 為多重抗藥性結核 (MDR-TB)
age = np.clip(rng.normal(48, 16, size=n), 15, 90).round().astype(int)

# 非抗藥性病人平均約 150 天痊癒；抗藥性病人療程長，平均約 320 天才痊癒
scale_cure = np.where(drug_resistant == 1, 320, 150)
duration_to_cure = rng.exponential(scale_cure)

follow_up_end = 540  # 追蹤 18 個月，超過此時間仍未痊癒者視為右設限（censored）
time_to_event = np.minimum(duration_to_cure, follow_up_end)
event = (duration_to_cure <= follow_up_end).astype(int)  # 1 = 痊癒, 0 = 追蹤結束仍未痊癒（設限）

tb = pd.DataFrame({
    "patient_id": [f"TB{i:04d}" for i in range(n)],
    "age": age,
    "drug_resistant": drug_resistant,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# KM 曲線：MDR-TB vs 非抗藥性
fig, ax = plt.subplots(figsize=(8, 5))

kmf_results = {}
for label, mask in [("MDR-TB（抗藥性）", tb["drug_resistant"] == 1),
                     ("非抗藥性", tb["drug_resistant"] == 0)]:
    sub = tb[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, 痊癒={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)
    kmf_results[label] = kmf

ax.set_title("結核病治療存活曲線：MDR-TB vs 非抗藥性（event = 痊癒）")
ax.set_xlabel("治療開始後天數")
ax.set_ylabel("尚未痊癒的機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Log-rank 檢定
resistant = tb[tb["drug_resistant"] == 1]
susceptible = tb[tb["drug_resistant"] == 0]

result = logrank_test(
    resistant["time_to_event"], susceptible["time_to_event"],
    event_observed_A=resistant["event"],
    event_observed_B=susceptible["event"],
)
print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

# 中位痊癒天數
for label, kmf in kmf_results.items():
    print(f"{label} 中位痊癒天數：{kmf.median_survival_time_:.1f} 天")

print("\n=== 解讀 ===")
if result.p_value < 0.05:
    print("→ p < 0.05：抗藥性結核（MDR-TB）病人的痊癒時間顯著較長")
    print("→ MDR-TB 需要更長的療程與更密集的照護，也提高治療中斷的風險")
else:
    print("→ 兩組痊癒時間差異未達統計顯著")

## 題目 5 解答

In [ ]:
# 資料：COVID-19 住院世代 -- 比較 ICU 收治 vs 一般病房病人住院至死亡的時間
rng = np.random.default_rng(519)
n = 500

age = np.clip(rng.normal(60, 18, size=n), 18, 95).round().astype(int)
is_male = rng.binomial(1, 0.5, size=n)
icu_admission = rng.binomial(1, 0.22, size=n)
diabetes = rng.binomial(1, 0.25, size=n)

baseline_hazard = 1 / 420  # 一般病房、未患糖尿病、60 歲病人的基準死亡風險
linear_pred = 1.0 * icu_admission + 0.4 * diabetes + 0.03 * (age - 60)
hazard = baseline_hazard * np.exp(linear_pred)
duration = rng.exponential(1 / hazard)

follow_up_end = 60  # 60 天追蹤期，超過此時間仍住院或已出院者視為右設限
time_to_event = np.minimum(duration, follow_up_end)
event = (duration <= follow_up_end).astype(int)  # 1 = 住院期間死亡, 0 = 出院或追蹤結束（設限）

covid = pd.DataFrame({
    "patient_id": [f"CV{i:04d}" for i in range(n)],
    "age": age,
    "is_male": is_male,
    "icu_admission": icu_admission,
    "diabetes": diabetes,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# KM 曲線：ICU vs 一般病房
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("ICU", covid["icu_admission"] == 1),
                     ("一般病房", covid["icu_admission"] == 0)]:
    sub = covid[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("COVID-19 住院存活曲線：ICU vs 一般病房")
ax.set_xlabel("住院後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank 檢定
icu_yes = covid[covid["icu_admission"] == 1]
icu_no = covid[covid["icu_admission"] == 0]

result = logrank_test(
    icu_yes["time_to_event"], icu_no["time_to_event"],
    event_observed_A=icu_yes["event"],
    event_observed_B=icu_no["event"],
)
print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

# Cox 迴歸
cph = CoxPHFitter()
cph.fit(covid[["time_to_event", "event", "age", "is_male", "icu_admission", "diabetes"]],
        duration_col="time_to_event", event_col="event")

print("\n=== Cox 迴歸結果 ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

print("\n=== 解讀 ===")
icu_hr = cph.summary.loc["icu_admission", "exp(coef)"]
print(f"icu_admission HR = {icu_hr:.3f}")
print("→ ICU 收治的 HR 顯著大於 1，但這不代表 ICU 治療本身『有害』")
print("→ ICU 是留給病情最嚴重病人的照護資源，這是典型的 confounding by indication")
print("→ 解讀觀察性資料的 HR 時，務必考慮『為什麼病人會被收入 ICU』這個問題")

## 題目 6 解答

In [ ]:
# 資料：麻疹 (measles) 病例接觸者 -- 比較接種疫苗 vs 未接種疫苗接觸者暴露後發病的時間
rng = np.random.default_rng(626)
n = 350

vaccinated = rng.binomial(1, 0.6, size=n)  # 60% 接觸者曾接種麻疹疫苗
age = np.clip(rng.normal(10, 8, size=n), 0, 60).round().astype(int)

# 未接種者感染機率高（侵襲率高）；接種者多數有保護力，突破感染機率低
p_infected = np.where(vaccinated == 1, 0.12, 0.85)
infected = rng.binomial(1, p_infected)

# 若感染，潛伏期（暴露到發病）約 10-14 天；未感染者在觀察期內不會發病
incubation = np.clip(rng.normal(12, 2.2, size=n), 5, 21)
surveillance_end = 21  # 接觸者追蹤 21 天

time_to_event = np.where(infected == 1, incubation, surveillance_end)
event = infected  # 1 = 觀察期內發病, 0 = 觀察期結束仍未發病（設限）

measles = pd.DataFrame({
    "contact_id": [f"MS{i:04d}" for i in range(n)],
    "age": age,
    "vaccinated": vaccinated,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# KM 曲線：未接種 vs 已接種
fig, ax = plt.subplots(figsize=(8, 5))

km_results = {}
for label, mask in [("未接種", measles["vaccinated"] == 0),
                     ("已接種", measles["vaccinated"] == 1)]:
    sub = measles[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, 發病={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)
    km_results[label] = kmf

ax.set_title("麻疹接觸者存活曲線：未接種 vs 已接種（event = 發病）")
ax.set_xlabel("暴露後天數")
ax.set_ylabel("尚未發病的機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank 檢定
unvax = measles[measles["vaccinated"] == 0]
vax = measles[measles["vaccinated"] == 1]

result = logrank_test(
    unvax["time_to_event"], vax["time_to_event"],
    event_observed_A=unvax["event"],
    event_observed_B=vax["event"],
)
print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

print(f"\n未接種組中位發病時間：{km_results['未接種'].median_survival_time_:.1f} 天（近似麻疹潛伏期中位數）")

print("\n=== 解讀 ===")
if result.p_value < 0.05:
    print("→ p < 0.05：接種疫苗顯著降低／延後接觸者發病")
    print("→ 未接種組的中位發病時間可作為麻疹潛伏期的參考，有助於設定接觸者追蹤與檢疫天數")
else:
    print("→ 兩組發病時間差異未達統計顯著")

## 題目 7 解答

In [ ]:
# 資料：登革熱 (dengue) 病例世代 -- 比較次發感染 vs 初次感染病人重症化的時間
rng = np.random.default_rng(727)
n = 450

secondary_infection = rng.binomial(1, 0.35, size=n)  # 35% 為次發感染（曾感染過不同型別登革病毒）
age = np.clip(rng.normal(32, 16, size=n), 1, 85).round().astype(int)
is_male = rng.binomial(1, 0.48, size=n)

baseline_hazard = 1 / 45  # 初次感染、32 歲病人的基準重症化風險
linear_pred = 1.1 * secondary_infection + 0.012 * (age - 32)
hazard = baseline_hazard * np.exp(linear_pred)
duration = rng.exponential(1 / hazard)

follow_up_end = 14  # 發病後 14 天臨床追蹤期
time_to_event = np.minimum(duration, follow_up_end)
event = (duration <= follow_up_end).astype(int)  # 1 = 進展為重症登革熱, 0 = 追蹤期結束仍未重症化（設限）

dengue = pd.DataFrame({
    "case_id": [f"DF{i:04d}" for i in range(n)],
    "age": age,
    "is_male": is_male,
    "secondary_infection": secondary_infection,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# KM 曲線：次發感染 vs 初次感染
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("次發感染", dengue["secondary_infection"] == 1),
                     ("初次感染", dengue["secondary_infection"] == 0)]:
    sub = dengue[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, 重症={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("登革熱重症化存活曲線：次發感染 vs 初次感染")
ax.set_xlabel("發病後天數")
ax.set_ylabel("尚未重症化的機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank 檢定
secondary = dengue[dengue["secondary_infection"] == 1]
primary = dengue[dengue["secondary_infection"] == 0]

result = logrank_test(
    secondary["time_to_event"], primary["time_to_event"],
    event_observed_A=secondary["event"],
    event_observed_B=primary["event"],
)
print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

# Cox 迴歸
cph = CoxPHFitter()
cph.fit(dengue[["time_to_event", "event", "age", "is_male", "secondary_infection"]],
        duration_col="time_to_event", event_col="event")

print("\n=== Cox 迴歸結果 ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

print("\n=== 解讀 ===")
hr = cph.summary.loc["secondary_infection", "exp(coef)"]
p_val = cph.summary.loc["secondary_infection", "p"]
print(f"secondary_infection HR = {hr:.3f}, p = {p_val:.4f}")
if p_val < 0.05 and hr > 1:
    print("→ 次發感染顯著提高重症化的風險比")
    print("→ 此結果與抗體依賴增強效應（ADE）的假說一致：")
    print("  非中和性的既有抗體可能協助病毒進入細胞，加重病程")
else:
    print("→ 次發感染對重症化風險的影響未達統計顯著")

## 題目 8 解答

In [ ]:
# 資料：延續本章開頭已載入的 cases（松柏護理之家退伍軍人症病例，event = 死亡）
cox_cols_q8 = ["time_to_event", "event", "age", "sex",
               "icu_admission", "immunosuppressed", "comorbidity_cancer"]
cox_df8 = cases[cox_cols_q8].copy()
cox_df8["is_male"] = (cox_df8["sex"] == "M").astype(int)
cox_df8 = cox_df8.drop(columns=["sex"])

print(cox_df8.describe().round(2))

# Cox 迴歸
cph8 = CoxPHFitter()
cph8.fit(cox_df8, duration_col="time_to_event", event_col="event")

print("\n=== Cox 迴歸結果 ===")
summary8 = cph8.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary8.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary8.round(3).to_string())

print("\n=== 比例風險假設檢查 ===")
try:
    cph8.check_assumptions(cox_df8, p_value_threshold=0.05, show_plots=False)
except Exception as exc:
    print(f"（比例風險假設檢定於小樣本下可能不穩定：{exc}）")

c_index = cph8.concordance_index_
print(f"\nC-index（一致性指標）= {c_index:.3f}")

print("\n=== 解讀 ===")
sig_vars = summary8[summary8["p_value"] < 0.05].index.tolist()
if sig_vars:
    print(f"→ 顯著提高死亡風險的因子：{', '.join(sig_vars)}")
else:
    print("→ 本模型中沒有變項達到統計顯著（p < 0.05）")
print("→ 樣本僅 19 例死亡卻放入 5 個共變項，每變項事件數（events per variable）偏低，")
print("  估計值的信賴區間可能很寬，模型容易過度配適，結果宜謹慎解讀")
if c_index > 0.7:
    print(f"→ C-index = {c_index:.3f} > 0.7，模型有不錯的區辨能力")
else:
    print(f"→ C-index = {c_index:.3f}，模型的區辨能力有限，仍需更多資料驗證")